# Day 9 • Text-to-GitHub AI Agent
**⚠️ Action Required:** You must replace `'YOUR_GITHUB_PERSONAL_ACCESS_TOKEN'` and `'YOUR_GEMINI_API_KEY'` with your actual API keys before running this cell so the script can authenticate and create the repository on your behalf.

In [ ]:
# ==========================================
# STEP 1: Install Required Libraries
# ==========================================
!pip install -q PyGithub google-generativeai

import os
import json
import google.generativeai as genai
from github import Github
from github import Auth

# ==========================================
# STEP 2: Configuration & Authentication
# ==========================================
# ⚠️ REPLACE THESE WITH YOUR ACTUAL API KEYS ⚠️
GITHUB_TOKEN = "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN" 
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"             

# Configure APIs
genai.configure(api_key=GEMINI_API_KEY)
auth = Auth.Token(GITHUB_TOKEN)
gh = Github(auth=auth)

# ==========================================
# STEP 3: LLM Intent Extraction & Scaffolding
# ==========================================
def parse_requirements_to_json(user_prompt):
    """
    Uses Gemini to parse natural language into a structured JSON payload
    containing repo details, README, and necessary scaffolding files.
    """
    model = genai.GenerativeModel('gemini-1.5-flash')
    
    system_prompt = """
    You are an AI Agent that converts natural language project descriptions into GitHub repository scaffolding.
    Extract the user's intent and return a strictly formatted JSON response. Do not include markdown formatting like ```json in the output, just the raw JSON object.
    
    The JSON must have this exact structure:
    {
        "repo_name": "A suitable lowercase-hyphenated-name",
        "description": "A short 1-sentence description of the repo",
        "readme": "Detailed Markdown content for README.md explaining the project",
        "files": [
            {
                "path": ".gitignore",
                "content": "appropriate gitignore content for the tech stack"
            },
            {
                "path": ".github/workflows/ci.yml",
                "content": "A basic GitHub Actions CI YAML file for the tech stack"
            },
            {
                "path": "src/main.py", 
                "content": "Basic starter code based on the prompt (adjust filename/path based on language)"
            }
        ]
    }
    """
    
    print("🤖 Agent: Analyzing your request and generating scaffolding...")
    response = model.generate_content(system_prompt + "\n\nUser Request: " + user_prompt)
    
    try:
        # Clean up potential markdown formatting from the response
        clean_text = response.text.strip().removeprefix('```json').removesuffix('```').strip()
        parsed_data = json.loads(clean_text)
        return parsed_data
    except json.JSONDecodeError as e:
        print("❌ Error: Failed to parse LLM output into JSON.")
        print("Raw Output:", response.text)
        return None

# ==========================================
# STEP 4: GitHub API Execution
# ==========================================
def create_github_repo_and_files(repo_data):
    """
    Automates the creation of the repository and uploads the scaffolding files via API.
    """
    user = gh.get_user()
    repo_name = repo_data['repo_name']
    
    print(f"🚀 Creating repository: {repo_name}...")
    try:
        # Create Repo
        repo = user.create_repo(
            name=repo_name,
            description=repo_data['description'],
            private=False,
            auto_init=True # Creates an initial commit so we can add files
        )
        print(f"✅ Repository created: {repo.html_url}")
        
        # Add README
        print("📝 Generating dynamic README.md...")
        repo.update_file(
            path="README.md",
            message="Initial commit: Add dynamic README generated by AI Agent",
            content=repo_data['readme'],
            sha=repo.get_contents("README.md").sha # overwrite the auto-init README
        )
        
        # Add Scaffolding Files
        for file in repo_data['files']:
            print(f"📁 Creating template file: {file['path']}...")
            repo.create_file(
                path=file['path'],
                message=f"Add template: {file['path']}",
                content=file['content']
            )
            
        print("\n🎉 All done! Your project is ready.")
        print(f"👉 Link: {repo.html_url}")
        
    except Exception as e:
        print(f"❌ GitHub API Error: {e}")

# ==========================================
# STEP 5: The Agent Loop
# ==========================================
def run_text_to_github_agent():
    print("=========================================")
    print("  TEXT-TO-GITHUB AI AGENT (Assignment 9) ")
    print("=========================================\n")
    
    # 1. Input
    user_prompt = input("Describe the project you want to create on GitHub: \n> ")
    
    # 2. Intent Extraction
    repo_data = parse_requirements_to_json(user_prompt)
    
    if not repo_data:
        return

    # 3. Confirmation
    print("\n=========================================")
    print("📄 PROPOSED PLAN:")
    print(f"Repo Name:   {repo_data['repo_name']}")
    print(f"Description: {repo_data['description']}")
    print(f"Files to add: README.md, " + ", ".join([f["path"] for f in repo_data['files']]))
    print("=========================================")
    
    confirm = input("\nDo you want to proceed and create this on GitHub? (yes/no): ").strip().lower()
    
    # 4. API Calls
    if confirm in ['yes', 'y']:
        create_github_repo_and_files(repo_data)
    else:
        print("🛑 Agent execution aborted by user.")

# ==========================================
# EXECUTE AGENT
# ==========================================
run_text_to_github_agent()
